# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [6]:
# Load the data
df = pd.read_csv('data/AviationData.csv', encoding='latin-1')

# Inspect the first 5 rows
df.head()

C:\Users\user\AppData\Local\Temp\ipykernel_20964\3846992832.py:2: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('data/AviationData.csv', encoding='latin-1')


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [7]:
# Check column names
print(df.columns)

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='object')


In [9]:
# Converting text date to real date
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')

# Keep only accidents from 1983 onwards
df = df[df['Event.Date'].dt.year >= 1983]

# Remove amateur/experimental aircraft
df = df[~df['Amateur.Built'].str.contains('Yes', na=False, case=False)]
df = df[~df['Aircraft.Category'].str.contains('Experimental|Amateur', na=False, case=False)]

# Remove rows missing critical information
df = df.dropna(subset=['Event.Date', 'Make', 'Model', 'Aircraft.damage'])

#Verify results 
print (f"Shape after cleaning: {df.shape}")
df[['Event.Date', 'Make', 'Model', 'Aircraft.Category', 'Amateur.Built']].head()

Shape after cleaning: (73898, 31)


,Event.Date,Make,Model,Aircraft.Category,Amateur.Built
3601,1983-01-01,Cessna,182P,NaN,No
3602,1983-01-01,Cessna,182RG,NaN,No
3603,1983-01-01,Cessna,182P,NaN,No
3604,1983-01-01,Piper,PA-28R-200,NaN,No
3605,1983-01-01,Cessna,140,NaN,No


In [10]:
# Create binary columns for safety outcomes
# 1. Total destruction: 1 if 'Destroyed' or 'Substantial', 0 otherwise
df['total_destruction'] = df['Aircraft.damage'].str.contains(
    'Destroyed|Substantial', na=False, case=False
).astype(int)

# 2. Fatal or serious injury: 1 if any fatal/serious injuries, 0 otherwise
df['fatal_or_serious'] = (
    (df['Total.Fatal.Injuries'] > 0) | 
    (df['Total.Serious.Injuries'] > 0)
).astype(int)

# 3. Create aircraft size category (small = <=9 seats, large = 10+ seats)
# Note: This dataset doesn't have seat count, so we'll use a proxy:
# Small = 'Small' in Aircraft.Category OR Make is typical small-plane brand
small_makes = ['Cessna', 'Piper', 'Beechcraft', 'Mooney', 'Cirrus']
df['size_category'] = 'large'  # default
df.loc[df['Make'].isin(small_makes), 'size_category'] = 'small'

# Verify new columns
print("New columns added: total_destruction, fatal_or_serious, size_category")
print(f"\nDestruction rate overall: {df['total_destruction'].mean():.2%}")
print(f"Injury rate overall: {df['fatal_or_serious'].mean():.2%}")
print(f"\nSample with new columns:")
df[['Make', 'Model', 'size_category', 'total_destruction', 'fatal_or_serious']].head()

New columns added: total_destruction, fatal_or_serious, size_category

Destruction rate overall: 96.32%
Injury rate overall: 31.38%

Sample with new columns:


,Make,Model,size_category,total_destruction,fatal_or_serious
3601,Cessna,182P,small,1,0
3602,Cessna,182RG,small,1,0
3603,Cessna,182P,small,1,0
3604,Piper,PA-28R-200,small,1,0
3605,Cessna,140,small,1,0


In [11]:
## 🧮 DERIVED METRIC: Injury Rate Per Person on Board
## Goal: Calculate the probability that a random person on the plane is hurt.
## Formula: (Fatal + Serious Injuries) / Total People on Board

# 1. Calculate 'Total Souls on Board'
# We sum up every category of people reported in the accident
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']

# Check if these columns exist before trying to sum them
if all(col in df.columns for col in injury_cols):
    df['total_on_board'] = df[injury_cols].sum(axis=1)
    
    # 2. Handle Missing Data (Imputation)
    # If we don't know how many people were on board, use the median (middle value)
    # This prevents us from deleting useful rows just because one number is missing
    median_passengers = df['total_on_board'].median()
    df['total_on_board'] = df['total_on_board'].fillna(median_passengers)
    
    # 3. Count the "Bad Outcomes" (Fatal + Serious only)
    # We fill NaN with 0 so the math works
    df['serious_harm_count'] = df['Total.Fatal.Injuries'].fillna(0) + df['Total.Serious.Injuries'].fillna(0)
    
    # 4. Calculate the Rate
    # We add 0.1 to the bottom so we never divide by zero (which causes errors)
    df['injury_rate_per_person'] = df['serious_harm_count'] / (df['total_on_board'] + 0.1)
    
    # 5. Safety Cap: Ensure rate is between 0 and 1
    df['injury_rate_per_person'] = df['injury_rate_per_person'].clip(0, 1)

else:
    # Fallback if columns are missing: just use the simple 0/1 flag
    df['injury_rate_per_person'] = df['fatal_or_serious']
    print("⚠️ Note: Using simple binary flag because passenger columns were missing.")

# 6. Verify the new metric
print(f"✅ New metric 'injury_rate_per_person' created.")
print(f"Average injury rate per person: {df['injury_rate_per_person'].mean():.2%}")
print("\nPreview:")
df[['Make', 'Model', 'serious_harm_count', 'total_on_board', 'injury_rate_per_person']].head()

✅ New metric 'injury_rate_per_person' created.
Average injury rate per person: 26.18%

Preview:


,Make,Model,serious_harm_count,total_on_board,injury_rate_per_person
3601,Cessna,182P,0.0,4.0,0.0
3602,Cessna,182RG,0.0,2.0,0.0
3603,Cessna,182P,0.0,1.0,0.0
3604,Piper,PA-28R-200,0.0,2.0,0.0
3605,Cessna,140,0.0,2.0,0.0


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [12]:
## AIRCRAFT DAMAGE CLEANING

# 1. Check what values exist in the Aircraft.damage column
print("Unique values in Aircraft.damage:")
print(df['Aircraft.damage'].value_counts())
print(f"\nMissing values: {df['Aircraft.damage'].isna().sum()}")

# 2. Create derived column: was_destroyed (1 = Yes, 0 = No)
# We consider "Destroyed" or "Substantial" damage as total destruction
df['was_destroyed'] = df['Aircraft.damage'].str.contains(
    'Destroyed|Substantial', 
    na=False,  # Don't count NaN as destroyed
    case=False  # Ignore case (destroyed vs Destroyed)
).astype(int)

# 3. Verify
print(f"\n✅ Created 'was_destroyed' column")
print(f"Destruction rate: {df['was_destroyed'].mean():.2%}")
print("\nSample:")
df[['Aircraft.damage', 'was_destroyed']].head(10)

Unique values in Aircraft.damage:
Aircraft.damage
Substantial    55689
Destroyed      15489
Minor           2604
Unknown          116
Name: count, dtype: int64

Missing values: 0

✅ Created 'was_destroyed' column
Destruction rate: 96.32%

Sample:


,Aircraft.damage,was_destroyed
3601,Substantial,1
3602,Substantial,1
3603,Substantial,1
3604,Substantial,1
3605,Substantial,1
3607,Destroyed,1
3608,Destroyed,1
3609,Destroyed,1
3610,Substantial,1
3611,Substantial,1


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

## Cleaning Tasks for 'Make' Column
1. **Standardize formatting**: Strip leading/trailing whitespace and convert to Title Case to fix inconsistencies (e.g., "cessna " vs "Cessna").
2. **Handle missing/invalid values**: Drop rows where the Make is missing or recorded as "nan".
3. **Filter for statistical robustness**: Keep only Makes with ≥50 recorded accidents to ensure comparisons are reliable and not skewed by tiny sample sizes.

In [13]:
# 1. Initial inspection
print("Top 10 Makes before cleaning:")
print(df['Make'].value_counts().head(10))
print(f"Missing 'Make' values: {df['Make'].isna().sum()}")

# 2. Clean formatting
df['Make'] = df['Make'].astype(str).str.strip().str.title()
df = df[df['Make'] != 'Nan']  # Remove string 'nan' if present
df = df.dropna(subset=['Make'])

# 3. Filter for Makes with >= 50 accidents (statistical robustness)
make_counts = df['Make'].value_counts()
valid_makes = make_counts[make_counts >= 50].index
df = df[df['Make'].isin(valid_makes)].copy()

# 4. Verify results
print(f"\n✅ 'Make' column cleaned and filtered")
print(f"Unique Makes remaining: {df['Make'].nunique()}")
print(f"Total rows after filtering: {len(df)}")
print("\nTop 10 Makes by accident count:")
print(df['Make'].value_counts().head(10))

Top 10 Makes before cleaning:
Make
Cessna     20699
Piper      11203
CESSNA      4809
Beech       4012
PIPER       2788
Bell        1968
Mooney      1024
BEECH       1016
Boeing      1012
Grumman      979
Name: count, dtype: int64
Missing 'Make' values: 0

✅ 'Make' column cleaned and filtered
Unique Makes remaining: 86
Total rows after filtering: 67674

Top 10 Makes by accident count:
Make
Cessna      25508
Piper       13991
Beech        5028
Bell         2535
Boeing       1574
Mooney       1261
Robinson     1177
Grumman      1055
Bellanca      973
Hughes        863
Name: count, dtype: int64


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

## Cleaning Tasks for 'Model' Column
1. **Remove missing values**: Drop rows where `Model` is NaN to ensure clean analysis.
2. **Standardize formatting**: Strip whitespace and convert to uppercase for consistency.
3. **Check uniqueness**: Determine if Model names are unique to each Make (e.g., is "172" only a Cessna, or do other Makes also use "172"?).
4. **Create unique identifier**: If Model names are NOT unique across Makes, create a new column `aircraft_type` = `Make` + "_" + `Model` (e.g., "Cessna_172") to uniquely identify each plane type.

In [15]:
# 1. Remove rows with missing Model
df = df.dropna(subset=['Model']).copy()

# 2. Clean formatting
df['Model'] = df['Model'].astype(str).str.strip().str.upper()

# 3. Check: Are Model names unique to each Make?
# Example: Does "172" appear with multiple Makes?
model_make_check = df.groupby('Model')['Make'].nunique()
non_unique_models = model_make_check[model_make_check > 1]

print(f"Total unique Models: {df['Model'].nunique()}")
print(f"Models used by multiple Makes: {len(non_unique_models)}")
if len(non_unique_models) > 0:
    print(f"\n Example: '{non_unique_models.index[0]}' is used by {model_make_check[non_unique_models.index[0]]} different Makes")

# 4. Create unique identifier: Make_Model (e.g., "Cessna_172")
df['aircraft_type'] = df['Make'] + "_" + df['Model']

# 5. Verify
print(f"\n Created 'aircraft_type' column")
print(f"Unique aircraft types: {df['aircraft_type'].nunique()}")
print("\nSample of aircraft_type:")
df[['Make', 'Model', 'aircraft_type']].head(10)

Total unique Models: 5561
Models used by multiple Makes: 411

 Example: '100' is used by 4 different Makes

 Created 'aircraft_type' column
Unique aircraft types: 6089

Sample of aircraft_type:


,Make,Model,aircraft_type
3601,Cessna,182P,Cessna_182P
3602,Cessna,182RG,Cessna_182RG
3603,Cessna,182P,Cessna_182P
3604,Piper,PA-28R-200,Piper_PA-28R-200
3605,Cessna,140,Cessna_140
3607,Cessna,340A,Cessna_340A
3608,North American,T-6G,North American_T-6G
3609,Piper,PA-24-250,Piper_PA-24-250
3610,Piper,PA-32-301R,Piper_PA-32-301R
3611,Beech,V-35B,Beech_V-35B


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

## Cleaning Tasks for Additional Columns
1. **Engine.Type**: Standardize text formatting, strip whitespace, replace placeholder strings ("Unknown", "nan") with proper NaN.
2. **Weather.Condition**: Clean formatting, ensure consistent capitalization for categories like VMC/IMC.
3. **Purpose.of.flight**: Standardize text, remove trailing spaces, handle missing/unknown entries.
4. **Broad.phase.of.flight**: Clean formatting, ensure consistent phase labels (e.g., "Takeoff", "Landing", "Cruise").
5. **Number.of.Engines**: Ensure numeric data type. Coerce any non-numeric entries to NaN so calculations won't break later.
*Note: Per instructions, we are standardizing and inspecting these columns but not imputing or dropping missing values at this stage.*

In [16]:
import numpy as np

# List of columns to clean
text_cols = ['Engine.Type', 'Weather.Condition', 'Purpose.of.flight', 'Broad.phase.of.flight']

# 1. Clean text-based columns
for col in text_cols:
    print(f"=== {col} ===")
    print(f"Missing before: {df[col].isna().sum()}")
    print("Top values before:")
    print(df[col].value_counts().head(3), "\n")
    
    # Cleaning: strip whitespace, standardize case, convert placeholder strings to NaN
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].replace(['Nan', 'Unknown', ''], np.nan)
    
    print("Top values after:")
    print(df[col].value_counts().head(3), "\n")
    print("-" * 40)

# 2. Clean Number.of.Engines (numeric column)
print("=== Number.of.Engines ===")
print(f"Missing before: {df['Number.of.Engines'].isna().sum()}")
print(f"Data type before: {df['Number.of.Engines'].dtype}")

# Ensure numeric type; force any weird text into NaN
df['Number.of.Engines'] = pd.to_numeric(df['Number.of.Engines'], errors='coerce')

print(f"Data type after: {df['Number.of.Engines'].dtype}")
print(f"Missing after: {df['Number.of.Engines'].isna().sum()}")
print("✅ All additional columns cleaned and inspected.")

=== Engine.Type ===
Missing before: 4458
Top values before:
Engine.Type
Reciprocating    54691
Turbo Shaft       2918
Turbo Prop        2616
Name: count, dtype: int64 

Top values after:
Engine.Type
Reciprocating    54691
Turbo Shaft       2918
Turbo Prop        2616
Name: count, dtype: int64 

----------------------------------------
=== Weather.Condition ===
Missing before: 2715
Top values before:
Weather.Condition
VMC    59049
IMC     5081
UNK      650
Name: count, dtype: int64 

Top values after:
Weather.Condition
Vmc    59049
Imc     5081
Unk      829
Name: count, dtype: int64 

----------------------------------------
=== Purpose.of.flight ===
Missing before: 3896
Top values before:
Purpose.of.flight
Personal         36579
Instructional     9322
Unknown           4773
Name: count, dtype: int64 

Top values after:
Purpose.of.flight
Personal              36579
Instructional          9322
Aerial Application     4121
Name: count, dtype: int64 

---------------------------------------

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

## Column Removal Strategy
1. **Inspect Missing Data**: Calculate the percentage of missing values for every column.
2. **Identify Candidates**: Flag any column where more than **50%** of the data is missing. These columns likely contain insufficient information for reliable analysis.
3. **Execute Removal**: Drop these high-missing columns from the dataframe to clean up the dataset.
4. **Verify**: Print the list of dropped columns and the final dataframe shape.

In [18]:
# 1. Calculate percentage of missing values for each column
missing_pct = df.isnull().mean() * 100

# 2. Display columns with significant missing data
print("Columns with > 10% missing values:")
print(missing_pct[missing_pct > 10].sort_values(ascending=False))
print("\n")

# 3. Define threshold and drop columns
# We will drop any column with > 50% missing data
threshold = 0.5 
cols_to_drop = df.columns[df.isnull().mean() > threshold]

print(f"Columns to drop (>50% missing): {list(cols_to_drop)}")

df = df.drop(columns=cols_to_drop)

# 4. Verify
print(f"\n Dropped {len(cols_to_drop)} columns.")
print(f"Final dataframe shape: {df.shape}")

Columns with > 10% missing values:
Airport.Code              42.681089
Airport.Name              39.982859
Broad.phase.of.flight     28.504300
Publication.Date          17.281378
Total.Serious.Injuries    14.714661
Total.Minor.Injuries      13.802938
Total.Fatal.Injuries      13.239944
Purpose.of.flight         12.809942
dtype: float64


Columns to drop (>50% missing): []

 Dropped 0 columns.
Final dataframe shape: (67674, 33)


### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

## Save Cleaned DataFrame to CSV
Exporting the cleaned dataset keeps our workflow modular. The Analysis Notebook can now load this pre-processed file directly, avoiding duplicate cleaning code.

**Summary of cleaning applied before export:**
- Filtered to accidents from `1983` onwards
- Removed amateur/experimental aircraft
- Standardized text columns (`Make`, `Model`, `Engine.Type`, etc.)
- Created derived metrics: `was_destroyed`, `injury_rate_per_person`, `aircraft_type`
- Inspected and removed columns with >50% missing values

In [20]:
# Save the cleaned DataFrame to CSV
# index=False prevents pandas from saving row numbers as a column
output_path = 'data/cleaned_aviation_data.csv'
df.to_csv(output_path, index=False)

print(f" Cleaned data saved to: {output_path}")

# Verify the file was created
import os
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"📁File size: {file_size_mb:.2f} MB")

# Quick sanity check: load first 2 rows to confirm structure
verify = pd.read_csv(output_path, nrows=2)
print(f"\n🔍 Verification - Shape: {verify.shape}")
print("Columns:", list(verify.columns)[:5], "...")

 Cleaned data saved to: data/cleaned_aviation_data.csv
📁File size: 17.99 MB

🔍 Verification - Shape: (2, 33)
Columns: ['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date', 'Location'] ...
